In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from ortools.sat.python import cp_model
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gradio as gr
import io
import threading
import queue

# Force-close any previous instances lingering in the Jupyter Notebook memory
gr.close_all()

# Base configuration
BASE_DATE = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0)
COST_UNIT_SCALE = 1_000

# Fleet starts empty; populated via file upload or manual injection
INITIAL_FLEET = {}

ALL_KNOWN_CATEGORIES = [
    "Downhole Logging, Survey & Test", "Wellhead Maintenance", "Slickline Services",
    "CTU Services", "Sand Control & Monitoring", "Perforation",
    "Well Repair & Maintenance", "Well Stimulation", "Artificial Lift Enhancement"
]


def parse_categories(cats_raw, all_known):
    if not isinstance(cats_raw, str):
        return []
    if ';' in cats_raw:
        return [c.strip() for c in cats_raw.split(';') if c.strip()]
    candidate_cats = []
    for cat in all_known:
        if cat in cats_raw:
            candidate_cats.append(cat)
    return candidate_cats


def _as_nonnegative_int(value, field_name):
    numeric_value = float(value)
    if not np.isfinite(numeric_value) or numeric_value < 0 or not numeric_value.is_integer():
        raise ValueError(f"{field_name} must be a non-negative whole number.")
    return int(numeric_value)


def _cost_kusd_to_units(cost_kusd):
    try:
        cost = Decimal(str(cost_kusd))
    except Exception as exc:
        raise ValueError("Daily Cost (kUSD) must be a positive number.") from exc

    if not cost.is_finite() or cost <= 0:
        raise ValueError("Daily Cost (kUSD) must be a positive number.")
    cost_units = int((cost * COST_UNIT_SCALE).to_integral_value(rounding=ROUND_HALF_UP))
    if cost_units < 1:
        raise ValueError("Daily Cost (kUSD) must be at least 0.001.")
    return cost_units


def _well_cost_units(duration_days, mob_demob_days, daily_cost_kusd):
    return (duration_days + mob_demob_days) * _cost_kusd_to_units(daily_cost_kusd)


def _validate_optimization_inputs(df_candidates, fleet_specs):
    required_columns = {"Well_ID", "Job Category", "Duration_Days", "BOPD", "Lat", "Lon"}
    missing_columns = required_columns.difference(df_candidates.columns)
    if missing_columns:
        raise ValueError(f"Dataset is missing required column(s): {', '.join(sorted(missing_columns))}.")
    if not fleet_specs:
        raise ValueError("Add or upload at least one rig before optimizing.")

    candidates = df_candidates.copy().reset_index(drop=True)
    if candidates["Well_ID"].isna().any() or candidates["Well_ID"].astype(str).str.strip().eq("").any():
        raise ValueError("Every well must have a non-empty Well_ID.")
    if candidates["Well_ID"].duplicated().any():
        raise ValueError("Well_ID values must be unique.")

    for column in ["Duration_Days", "BOPD", "Lat", "Lon"]:
        candidates[column] = pd.to_numeric(candidates[column], errors="raise")
        if not np.isfinite(candidates[column]).all():
            raise ValueError(f"{column} contains an invalid number.")
    candidates["Duration_Days"] = candidates["Duration_Days"].map(
        lambda value: _as_nonnegative_int(value, "Duration_Days")
    )
    candidates["BOPD"] = candidates["BOPD"].map(
        lambda value: _as_nonnegative_int(value, "BOPD")
    )
    if ((candidates["Lat"] < -90) | (candidates["Lat"] > 90)).any():
        raise ValueError("Latitude must be between -90 and 90.")
    if ((candidates["Lon"] < -180) | (candidates["Lon"] > 180)).any():
        raise ValueError("Longitude must be between -180 and 180.")

    normalized_fleet = {}
    default_colors = ["#1f77b4", "#ff7f0e", "#d62728", "#2ca02c", "#9467bd", "#e377c2"]
    for idx, (rig_name, rig) in enumerate(fleet_specs.items()):
        categories = rig.get("Supported_Job_Categories") or []
        if not categories:
            raise ValueError(f"{rig_name} must support at least one job category.")
        color = rig.get("Color") or default_colors[idx % len(default_colors)]
        normalized_fleet[rig_name] = {
            **rig,
            "Mob_Demob_Days": _as_nonnegative_int(rig.get("Mob_Demob_Days"), f"{rig_name} Mob/Demob Days"),
            "Daily_Cost_kUSD": float(Decimal(str(rig.get("Daily_Cost_kUSD")))),
            "Supported_Job_Categories": list(categories),
            "Contract_End_Date": rig.get("Contract_End_Date", ""),
            "Color": color
        }
        _cost_kusd_to_units(normalized_fleet[rig_name]["Daily_Cost_kUSD"])

    return candidates, normalized_fleet

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

def generate_synthetic_wells(n_wells=20):
    np.random.seed(42)
    data = []
    for i in range(1, n_wells + 1):
        jcat = np.random.choice(ALL_KNOWN_CATEGORIES)
        dur = np.random.randint(5, 30)
        gain = np.random.randint(200, 1500)
        lat = np.random.uniform(3.5, 4.8)
        lon = np.random.uniform(111.0, 113.5)
        data.append({
            "Well_ID": f"WELL-{i:02d}", "Job Category": jcat,
            "Duration_Days": dur, "BOPD": gain,
            "Lat": lat, "Lon": lon
        })
    return pd.DataFrame(data)

# --- MONTE CARLO SIMULATION ENGINE ---
def generate_mc_plot(scheduled_df, fleet_specs, n_iterations=10000):
    """Runs a Monte Carlo simulation on the planned schedule, returning both the Plotly figure and iteration DataFrame."""
    if scheduled_df is None or scheduled_df.empty:
        return go.Figure(), pd.DataFrame()
    
    np.random.seed(42)
    sim_gains = []
    sim_costs = []
    
    for _ in range(n_iterations):
        iter_gain = 0
        iter_cost = 0
        for _, row in scheduled_df.iterrows():
            base_dur = row["Duration_Days"]
            base_gain = row["BOPD"]
            rig = row["Rig_Assigned"]
            mob = fleet_specs[rig]["Mob_Demob_Days"]
            rate = fleet_specs[rig]["Daily_Cost_kUSD"]
            
            # Triangular Risk Dist: Durations skew longer (0.8x to 1.5x)
            samp_dur = np.random.triangular(0.8 * base_dur, base_dur, 1.5 * base_dur)
            # Triangular Risk Dist: Gains skew lower (0.7x to 1.2x)
            samp_gain = np.random.triangular(0.7 * base_gain, base_gain, 1.2 * base_gain)
            
            iter_gain += samp_gain
            iter_cost += (samp_dur + mob) * rate
            
        sim_gains.append(iter_gain)
        sim_costs.append(iter_cost)

    # Build raw iteration dataset for export
    mc_iterations_df = pd.DataFrame({
        "Iteration": range(1, n_iterations + 1),
        "Simulated_Gain_BOPD": sim_gains,
        "Simulated_Cost_kUSD": sim_costs
    })

    # In Oil & Gas Reserves/Volume: P90 is high confidence (lowest), P10 is low confidence (highest)
    p90_g, p50_g, p10_g = np.percentile(sim_gains, [10, 50, 90]) 
    
    # In Oil & Gas Cost/Time: P90 is conservative estimate (highest), P10 is optimistic (lowest)
    p10_c, p50_c, p90_c = np.percentile(sim_costs, [10, 50, 90])
    
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Total Expected Gain (BOPD)", "Total Expected Cost (kUSD)"))
    
    fig.add_trace(go.Histogram(x=sim_gains, name="Gain", marker_color='green', opacity=0.7), row=1, col=1)
    fig.add_vline(x=p90_g, line_dash="dash", line_color="red", annotation_text=f"P90: {p90_g:,.0f}", row=1, col=1)
    fig.add_vline(x=p50_g, line_dash="dash", line_color="black", annotation_text=f"P50: {p50_g:,.0f}", row=1, col=1)
    fig.add_vline(x=p10_g, line_dash="dash", line_color="blue", annotation_text=f"P10: {p10_g:,.0f}", row=1, col=1)
    
    fig.add_trace(go.Histogram(x=sim_costs, name="Cost", marker_color='orange', opacity=0.7), row=1, col=2)
    fig.add_vline(x=p10_c, line_dash="dash", line_color="blue", annotation_text=f"P10: ${p10_c:,.0f}k", row=1, col=2)
    fig.add_vline(x=p50_c, line_dash="dash", line_color="black", annotation_text=f"P50: ${p50_c:,.0f}k", row=1, col=2)
    fig.add_vline(x=p90_c, line_dash="dash", line_color="red", annotation_text=f"P90: ${p90_c:,.0f}k", row=1, col=2)
    
    fig.update_layout(height=400, showlegend=False, template="plotly_white", margin=dict(t=40, b=10, l=10, r=10))
    return fig, mc_iterations_df


# --- CORE OPTIMIZATION ENGINE (WITH LIVE SEARCH STREAMING) ---
def run_optimization(df_candidates, fleet_specs, dropped_wells, locked_wells, prev_schedule):
    dropped_wells = dropped_wells or []
    locked_wells = locked_wells or []

    try:
        df_candidates, fleet_specs = _validate_optimization_inputs(df_candidates, fleet_specs)
    except (ValueError, TypeError, InvalidOperation) as exc:
        yield f"⚠️ Invalid optimization input: {exc}", go.Figure(), go.Figure(), go.Figure(), pd.DataFrame(), pd.DataFrame()
        return

    dropped_wells = set(dropped_wells)
    locked_wells = set(locked_wells).difference(dropped_wells)

    model = cp_model.CpModel()
    n_wells = len(df_candidates)
    horizon_days = 365  
    rig_names = list(fleet_specs.keys())
    n_rigs = len(rig_names)

    incompatible_wells = [
        row["Well_ID"]
        for _, row in df_candidates.iterrows()
        if row["Well_ID"] not in dropped_wells
        and not any(row["Job Category"] in fleet_specs[rig]["Supported_Job_Categories"] for rig in rig_names)
    ]
    if incompatible_wells:
        names = ", ".join(map(str, incompatible_wells))
        yield f"⚠️ No rig supports: {names}.", go.Figure(), go.Figure(), go.Figure(), pd.DataFrame(), pd.DataFrame()
        return
    if df_candidates[~df_candidates["Well_ID"].isin(dropped_wells)].empty:
        yield "No active wells to schedule.", go.Figure(), go.Figure(), go.Figure(), pd.DataFrame(), pd.DataFrame()
        return

    performed, starts, ends, assigned_rig, actual_ends = {}, {}, {}, {}, []
    rig_active_vars = {}

    for i in range(n_wells):
        performed[i] = model.NewBoolVar(f"performed_{i}")
        starts[i] = model.NewIntVar(0, horizon_days, f"start_{i}")
        ends[i] = model.NewIntVar(0, horizon_days, f"end_{i}")
        assigned_rig[i] = model.NewIntVar(0, n_rigs - 1, f"rig_{i}")
        
        well_id = df_candidates.loc[i, "Well_ID"]
        duration = int(df_candidates.loc[i, "Duration_Days"])
        jcat = df_candidates.loc[i, "Job Category"]

        if well_id in dropped_wells:
            model.Add(performed[i] == 0)
            continue  

        model.Add(performed[i] == 1)
        model.Add(starts[i] >= 0).OnlyEnforceIf(performed[i])

        rig_actives = []
        for r in range(n_rigs):
            rig_name = rig_names[r]
            supported_cats = fleet_specs[rig_name]["Supported_Job_Categories"]
            mob_demob = fleet_specs[rig_name]["Mob_Demob_Days"]
            total_well_time = duration + mob_demob
            
            rig_active = model.NewBoolVar(f"well_{i}_on_rig_{r}")
            rig_actives.append(rig_active)
            rig_active_vars[(i, r)] = rig_active
            
            if jcat not in supported_cats:
                model.Add(rig_active == 0)
            else:
                model.AddImplication(rig_active, performed[i])
                model.Add(assigned_rig[i] == r).OnlyEnforceIf(rig_active)
                model.Add(ends[i] == starts[i] + total_well_time).OnlyEnforceIf(rig_active)
                
        model.AddExactlyOne(rig_actives).OnlyEnforceIf(performed[i])
        
        actual_end = model.NewIntVar(0, horizon_days, f"actual_end_{i}")
        model.Add(actual_end == ends[i]).OnlyEnforceIf(performed[i])
        model.Add(actual_end == 0).OnlyEnforceIf(performed[i].Not())
        actual_ends.append(actual_end)

    # --- UNLOCKED ROUTING ---
    eligible_indices = [
        i for i in range(n_wells)
        if df_candidates.loc[i, "Well_ID"] not in dropped_wells
    ]
    node_by_well = {well_index: node_index + 1 for node_index, well_index in enumerate(eligible_indices)}
    transit_distance_terms = []

    for r, rig_name in enumerate(rig_names):
        route_arcs = []
        active_on_rig = [rig_active_vars[(i, r)] for i in eligible_indices]

        empty_route = model.NewBoolVar(f"rig_{r}_empty_route")
        model.Add(sum(active_on_rig) == 0).OnlyEnforceIf(empty_route)
        model.Add(sum(active_on_rig) >= 1).OnlyEnforceIf(empty_route.Not())
        route_arcs.append((0, 0, empty_route))

        efficiency_data = {}
        for i in eligible_indices:
            node_i = node_by_well[i]
            active_i = rig_active_vars[(i, r)]
            inactive_i = model.NewBoolVar(f"well_{i}_inactive_on_rig_{r}")
            model.Add(inactive_i + active_i == 1)
            route_arcs.append((node_i, node_i, inactive_i))

            first_arc = model.NewBoolVar(f"rig_{r}_first_well_{i}")
            last_arc = model.NewBoolVar(f"rig_{r}_last_well_{i}")
            model.AddImplication(first_arc, active_i)
            model.AddImplication(last_arc, active_i)
            route_arcs.extend([(0, node_i, first_arc), (node_i, 0, last_arc)])

            if df_candidates.loc[i, "Job Category"] in fleet_specs[rig_name]["Supported_Job_Categories"]:
                efficiency_data[i] = True

        for i in efficiency_data:
            for j in efficiency_data:
                if i == j:
                    continue

                distance_km = haversine(
                    df_candidates.loc[i, "Lat"], df_candidates.loc[i, "Lon"],
                    df_candidates.loc[j, "Lat"], df_candidates.loc[j, "Lon"],
                )
                transit_days = int(np.ceil(distance_km / 50.0)) + 1
                route_arc = model.NewBoolVar(f"rig_{r}_well_{i}_to_well_{j}")
                model.AddImplication(route_arc, rig_active_vars[(i, r)])
                model.AddImplication(route_arc, rig_active_vars[(j, r)])
                model.Add(starts[j] >= ends[i] + transit_days).OnlyEnforceIf(route_arc)
                route_arcs.append((node_by_well[i], node_by_well[j], route_arc))
                transit_distance_terms.append(route_arc * int(round(distance_km * 100)))

        model.AddCircuit(route_arcs)

    # --- TIME-TO-FIRST-OIL OBJECTIVE FUNCTION ---
    cumulative_oil_vars = []
    for i in eligible_indices:
        gain = int(df_candidates.loc[i, "BOPD"])
        production_days = model.NewIntVar(0, horizon_days, f"prod_days_{i}")
        model.Add(production_days == horizon_days - ends[i]).OnlyEnforceIf(performed[i])
        model.Add(production_days == 0).OnlyEnforceIf(performed[i].Not())
        
        cumulative_oil = model.NewIntVar(0, horizon_days * gain, f"cum_oil_{i}")
        model.AddMultiplicationEquality(cumulative_oil, [production_days, gain])
        cumulative_oil_vars.append(cumulative_oil)

    total_oil_produced = sum(cumulative_oil_vars) if cumulative_oil_vars else 0
    total_transit_units = sum(transit_distance_terms) if transit_distance_terms else 0
    
    model.Maximize((total_oil_produced * 100) - total_transit_units)

    # --- LIVE SEARCH CALLBACK VIA QUEUE & THREAD ---
    q = queue.Queue()

    class RouteSearchCallback(cp_model.CpSolverSolutionCallback):
        def __init__(self):
            cp_model.CpSolverSolutionCallback.__init__(self)
            self.solution_count = 0

        def on_solution_callback(self):
            self.solution_count += 1
            results = []
            for i in range(n_wells):
                if df_candidates.loc[i, "Well_ID"] in dropped_wells or not self.Value(performed[i]):
                    continue
                start_val = self.Value(starts[i])
                end_val = self.Value(ends[i])
                rig_val = self.Value(assigned_rig[i])
                rig_n = rig_names[rig_val]
                duration = int(df_candidates.loc[i, "Duration_Days"])
                mob_demob = fleet_specs[rig_n]["Mob_Demob_Days"]
                cost = _well_cost_units(duration, mob_demob, fleet_specs[rig_n]["Daily_Cost_kUSD"]) / COST_UNIT_SCALE
                gain = int(df_candidates.loc[i, "BOPD"])
                bopd_per_cost = round(gain / cost, 4) if cost > 0 else 0.0
                
                results.append({
                    "Well_ID": df_candidates.loc[i, "Well_ID"],
                    "Rig_Assigned": rig_n,
                    "Job Category": df_candidates.loc[i, "Job Category"],
                    "Start_Day": start_val, "End_Day": end_val,
                    "Start_Date": (BASE_DATE + timedelta(days=start_val)).strftime("%Y-%m-%d"),
                    "End_Date": (BASE_DATE + timedelta(days=end_val)).strftime("%Y-%m-%d"),
                    "Duration_Days": duration,
                    "Mob_Demob_Days": mob_demob,
                    "Lat": df_candidates.loc[i, "Lat"], "Lon": df_candidates.loc[i, "Lon"],
                    "BOPD": gain,
                    "Cost_kUSD": cost,
                    "BOPD_per_kUSD": bopd_per_cost,
                    "Contract_End_Date": fleet_specs[rig_n].get("Contract_End_Date", "N/A")
                })
            if results:
                sched_df = pd.DataFrame(results).sort_values(by=["Rig_Assigned", "Start_Day"])
                q.put(("searching", sched_df))

    cb = RouteSearchCallback()

    def solve_thread():
        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = 45
        solver.parameters.num_search_workers = 4
        status = solver.Solve(model, cb)
        q.put(("finished", status, solver))

    t = threading.Thread(target=solve_thread, daemon=True)
    t.start()

    final_status = None
    final_solver = None

    while True:
        try:
            msg_type, *data = q.get(timeout=0.1)
            if msg_type == "searching":
                sched_df = data[0]
                kpi, gantt, map_fig, mc_fig, mc_df, export_df = render_visuals(
                    sched_df, df_candidates, fleet_specs, None, None, 
                    "🔄 Optimizing... (Live Search Active, Please Wait)"
                )
                yield kpi, gantt, map_fig, mc_fig, mc_df, export_df
            elif msg_type == "finished":
                final_status, final_solver = data[0], data[1]
                break
        except queue.Empty:
            if not t.is_alive():
                break

    t.join()

    if final_status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        yield "⚠️ Configuration too constrained.", go.Figure(), go.Figure(), go.Figure(), pd.DataFrame(), pd.DataFrame()
        return

    results = []
    for i in range(n_wells):
        if df_candidates.loc[i, "Well_ID"] in dropped_wells or not final_solver.Value(performed[i]):
            continue
        start_val = final_solver.Value(starts[i])
        end_val = final_solver.Value(ends[i])
        rig_val = final_solver.Value(assigned_rig[i])
        rig_n = rig_names[rig_val]
        duration = int(df_candidates.loc[i, "Duration_Days"])
        mob_demob = fleet_specs[rig_n]["Mob_Demob_Days"]
        cost = _well_cost_units(duration, mob_demob, fleet_specs[rig_n]["Daily_Cost_kUSD"]) / COST_UNIT_SCALE
        gain = int(df_candidates.loc[i, "BOPD"])
        bopd_per_cost = round(gain / cost, 4) if cost > 0 else 0.0
        
        results.append({
            "Well_ID": df_candidates.loc[i, "Well_ID"],
            "Rig_Assigned": rig_n,
            "Job Category": df_candidates.loc[i, "Job Category"],
            "Start_Day": start_val, "End_Day": end_val,
            "Start_Date": (BASE_DATE + timedelta(days=start_val)).strftime("%Y-%m-%d"),
            "End_Date": (BASE_DATE + timedelta(days=end_val)).strftime("%Y-%m-%d"),
            "Duration_Days": duration,
            "Mob_Demob_Days": mob_demob,
            "Lat": df_candidates.loc[i, "Lat"], "Lon": df_candidates.loc[i, "Lon"],
            "BOPD": gain,
            "Cost_kUSD": cost,
            "BOPD_per_kUSD": bopd_per_cost,
            "Contract_End_Date": fleet_specs[rig_n].get("Contract_End_Date", "N/A")
        })

    if not results:
        yield render_visuals(pd.DataFrame(), df_candidates, fleet_specs, final_status, final_solver, "Time-to-First-Oil optimized schedule")
        return

    scheduled_df = pd.DataFrame(results).sort_values(by=["Rig_Assigned", "Start_Day"])
    yield render_visuals(scheduled_df, df_candidates, fleet_specs, final_status, final_solver, "Time-to-First-Oil optimized schedule")


# --- WRAPPER FOR DROPPED WELLS POPUP/RECOMMENDATION ---
def run_optimization_wrapper(df_candidates, fleet_specs, dropped_wells, locked_wells, prev_schedule, is_approved):
    """Intercepts execution to require user approval if wells are dropped."""
    if dropped_wells and not is_approved:
        dropped_str = ", ".join(dropped_wells)
        msg = f"⚠️ **Optimization Recommendation:** You are dropping {dropped_str}. Do you accept replacing them in the schedule to maximize overall efficiency for the remaining assets?"
        yield (
            gr.update(visible=False), 
            gr.update(visible=True),  
            gr.update(value=msg),     
            gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update()
        )
        return

    for kpi, gantt, map_fig, mc_fig, mc_df, export_df in run_optimization(df_candidates, fleet_specs, dropped_wells, locked_wells, prev_schedule):
        yield (
            gr.update(visible=True),  
            gr.update(visible=False), 
            gr.update(),              
            kpi, gantt, map_fig, mc_fig, mc_df, export_df
        )

# --- VISUAL RENDERING ---
def render_visuals(scheduled_df, df_candidates, fleet_specs, status=None, solver=None, optimization_mode=None):
    if scheduled_df.empty:
        return "No active schedule.", go.Figure(), go.Figure(), go.Figure(), pd.DataFrame(), pd.DataFrame()

    scheduled_df = scheduled_df.sort_values(by=["Rig_Assigned", "Start_Date"]).copy()
    scheduled_df["Efficiency_Rank"] = scheduled_df.groupby("Rig_Assigned").cumcount() + 1
    
    scheduled_df["Start_Dt"] = pd.to_datetime(scheduled_df["Start_Date"])
    scheduled_df["End_Dt"] = pd.to_datetime(scheduled_df["End_Date"])
    
    scheduled_df["Start_Str"] = scheduled_df["Start_Dt"].dt.strftime("%Y-%m-%d")
    scheduled_df["End_Str"] = scheduled_df["End_Dt"].dt.strftime("%Y-%m-%d")
    
    total_transit = 0
    map_fig = go.Figure()

    unassigned = df_candidates[~df_candidates["Well_ID"].isin(scheduled_df["Well_ID"])]
    if not unassigned.empty:
        map_fig.add_trace(go.Scattermap(
            lat=unassigned['Lat'], lon=unassigned['Lon'], mode='markers',
            marker=dict(size=7, color='gray', opacity=0.4),
            name="Candidate Wells", text=unassigned['Well_ID'],
        ))

    distances_from_prev = []
    for rig in scheduled_df["Rig_Assigned"].unique():
        rig_df = scheduled_df[scheduled_df["Rig_Assigned"] == rig].sort_values(by="Start_Dt").copy()
        rig_df['Seq'] = range(1, len(rig_df) + 1)
        color = fleet_specs.get(rig, {}).get("Color", "#1f77b4")
        
        map_fig.add_trace(go.Scattermap(
            lat=rig_df['Lat'], lon=rig_df['Lon'], mode='markers+lines+text',
            text=rig_df['Seq'].astype(str), textposition="top right",
            marker=dict(size=12, color=color), line=dict(width=2, color=color),
            name=rig
        ))

        prev_lat, prev_lon = None, None
        for _, row in rig_df.iterrows():
            if prev_lat is None:
                distances_from_prev.append(0.0)
            else:
                d = haversine(prev_lat, prev_lon, row["Lat"], row["Lon"])
                distances_from_prev.append(round(d, 1))
                total_transit += round(d, 1)
            prev_lat, prev_lon = row["Lat"], row["Lon"]

    scheduled_df["Transit_Dist_km"] = distances_from_prev

    map_fig.update_layout(map_style="carto-positron", map=dict(zoom=5.5, center=dict(lat=4.1, lon=112.2)), margin={"r":0,"t":40,"l":0,"b":0})

    gantt_fig = px.timeline(
        scheduled_df, x_start="Start_Dt", x_end="End_Dt", y="Rig_Assigned", color="Job Category", text="Well_ID",
        custom_data=["Well_ID", "Job Category", "BOPD", "Cost_kUSD", "Contract_End_Date", "Start_Str", "End_Str"]
    )
    
    gantt_fig.update_traces(
        textangle=90, 
        textposition="inside", 
        insidetextanchor="middle",
        texttemplate="%{text}<br>%{customdata[2]:,.0f} BOPD",
        hovertemplate=(
            "<b>Well:</b> %{customdata[0]}<br>"
            "<b>Category:</b> %{customdata[1]}<br>"
            "<b>Start:</b> %{customdata[5]}<br>"
            "<b>End:</b> %{customdata[6]}<br>"
            "<b>Gain:</b> %{customdata[2]} BOPD<br>"
            "<b>Cost:</b> $%{customdata[3]:,.1f}k<br>"
            "<b>Contract Ends:</b> %{customdata[4]}"
            "<extra></extra>"
        )
    )

    for rig_name, specs in fleet_specs.items():
        contract_end = specs.get("Contract_End_Date")
        if contract_end:
            gantt_fig.add_trace(go.Scatter(
                x=[contract_end, contract_end],
                y=[rig_name, rig_name],
                mode="markers+text",
                marker=dict(symbol="line-ns-open", size=18, color="red", line=dict(width=3)),
                text=[f"Contract End ({contract_end})", ""],
                textposition="top center",
                name=f"{rig_name} Contract",
                showlegend=False
            ))

    gantt_fig.update_yaxes(autorange="reversed")
    gantt_fig.update_layout(height=350, margin={"r":0,"t":10,"l":0,"b":0})
    
    # Run the Monte Carlo Simulation logic and retrieve iteration dataset
    mc_fig, mc_df = generate_mc_plot(scheduled_df, fleet_specs)

    scheduled_df["Start_Date"] = scheduled_df["Start_Str"]
    scheduled_df["End_Date"] = scheduled_df["End_Str"]

    if solver:
        if status == cp_model.OPTIMAL:
            status_txt = f"Optimal {optimization_mode}" if optimization_mode else "Optimal schedule"
        elif optimization_mode:
            status_txt = f"Feasible {optimization_mode}"
        else:
            status_txt = "Feasible schedule"
    else:
        status_txt = optimization_mode if optimization_mode else "Manual Human Override Active"

    total_cost = scheduled_df["Cost_kUSD"].sum()
    total_gain = scheduled_df["BOPD"].sum()
    schedule_efficiency = total_gain / total_cost if total_cost > 0 else 0.0

    kpi = (
        f"**Status:** {status_txt} | **Served:** {len(scheduled_df)} | "
        f"**Total Cost:** ${total_cost:,.3f}k | **Total Gain:** {total_gain:,} BOPD | "
        f"**Efficiency:** {schedule_efficiency:,.4f} BOPD/kUSD | "
        f"**Transit:** {total_transit:.1f} km"
    )

    export_df = scheduled_df[[
        "Well_ID", "Rig_Assigned", "Start_Date", "End_Date", "Job Category", 
        "Duration_Days","Transit_Dist_km", 
        "Cost_kUSD", "BOPD",
    ]]

    if not scheduled_df.empty and solver:
        for rig in scheduled_df["Rig_Assigned"].unique():
            rig_df = scheduled_df[scheduled_df["Rig_Assigned"] == rig]
            max_date = pd.to_datetime(rig_df["End_Date"].max())
            contract_end_str = fleet_specs.get(rig, {}).get("Contract_End_Date")
            
            if contract_end_str:
                try:
                    contract_end = pd.to_datetime(contract_end_str)
                    days_until_expire = (contract_end - max_date).days
                    if days_until_expire < 0:
                        gr.Warning(f"🚨 {rig} ALERT: Scheduled operations extend beyond contract end date ({contract_end_str})!")
                    elif days_until_expire <= 90:
                        gr.Warning(f"⚠️ {rig} TIMELINE WARNING: Contract expires on {contract_end_str}, within 3 months of the final scheduled operation. Recommend renewal.")
                except Exception:
                    pass

    return kpi, gantt_fig, map_fig, mc_fig, mc_df, export_df

# --- CSV EXPORT HELPER ---
def export_mc_csv(mc_df):
    if mc_df is None or mc_df.empty:
        gr.Warning("⚠️ No Monte Carlo simulation dataset available to export yet. Run optimization first.")
        return None
    path = "monte_carlo_iterations.csv"
    mc_df.to_csv(path, index=False)
    gr.Info("✅ Monte Carlo dataset exported successfully!")
    return path

# --- GRADIO LAYOUT ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    candidate_state = gr.State(generate_synthetic_wells(20))
    fleet_state = gr.State(INITIAL_FLEET)
    mc_data_state = gr.State(pd.DataFrame())

    gr.Markdown("# AI Rig Scheduling Optimization")
    
    with gr.Row():
        with gr.Column(scale=1):
            
            # 1. Dataset Ingestion & Templates
            with gr.Accordion("📂 1. Ingest Dataset", open=True):
                gr.Markdown("### Well Dataset")
                file_upload_wells = gr.File(label="Upload Well CSV/Excel", file_types=[".csv", ".xlsx"])
                download_well_template_btn = gr.DownloadButton("📥 Download Well Template", size="sm")
                
                gr.Markdown("### Rig Fleet Dataset")
                file_upload_rigs = gr.File(label="Upload Rig CSV/Excel", file_types=[".csv", ".xlsx"])
                download_rig_template_btn = gr.DownloadButton("📥 Download Rig Template", size="sm")

            # 2. Manual Well Entry
            with gr.Accordion("✍️ 2. Manual Well Entry", open=False):
                new_id = gr.Textbox(label="Well ID", value="WELL-21")
                new_cat = gr.Dropdown(choices=ALL_KNOWN_CATEGORIES, label="Job Category", value="Wellhead Maintenance")
                with gr.Row():
                    new_dur = gr.Number(label="Duration (Days)", value=10)
                    new_gain = gr.Number(label="Gain (BOPD)", value=800)
                with gr.Row():
                    new_lat = gr.Number(label="Latitude", value=4.20)
                    new_lon = gr.Number(label="Longitude", value=112.50)
                add_well_btn = gr.Button("Inject Well to Database", size="sm")

            # 3. Customize Rig Fleet
            with gr.Accordion("🚢 3. Customize Rig Fleet", open=False):
                new_rig_name = gr.Textbox(label="Rig Name", value="Rig-1 (Light Unit)")
                new_rig_type = gr.Dropdown(choices=["Light Unit", "Medium Unit", "Heavy Unit"], value="Light Unit")
                new_rig_cats = gr.Dropdown(choices=ALL_KNOWN_CATEGORIES, multiselect=True, label="Supported Categories")
                with gr.Row():
                    new_rig_mob = gr.Number(label="Mob/Demob Days", value=2)
                    new_rig_cost = gr.Number(label="Daily Cost (kUSD)", value=12)
                new_rig_contract = gr.Textbox(label="Contract End Date (YYYY-MM-DD)", value=(BASE_DATE + timedelta(days=365)).strftime("%Y-%m-%d"))
                add_rig_btn = gr.Button("Add Rig to Fleet", size="sm")

            # 4. Execution Engine
            with gr.Accordion("⚙️ 4. Filtering System", open=False):
                dropped_wells = gr.Dropdown(choices=[], multiselect=True, label="❌ Drop Wells")
                locked_wells = gr.Dropdown(choices=[], multiselect=True, label="🔒 Lock Wells")

            run_btn = gr.Button("Run Optimization", variant="primary", size="lg")
            
            with gr.Group(visible=False) as confirm_group:
                confirm_msg = gr.Markdown("")
                with gr.Row():
                    approve_btn = gr.Button("✅ Accept Recommendation", variant="primary")
                    deny_btn = gr.Button("❌ Deny & Reset Drops", variant="secondary")

            is_approved_state = gr.State(False)

        with gr.Column(scale=3):
            kpi_output = gr.Markdown("### Ready to optimize...")
            
            with gr.Accordion("📊 Fleet Sequence Gantt Timeline", open=True):
                gantt_output = gr.Plot()
            with gr.Accordion("🗺️ Fleet Route Map", open=False):
                map_output = gr.Plot()
            with gr.Accordion("🎲 Monte Carlo Risk Analysis (P10, P50, P90)", open=False):
                mc_output = gr.Plot()
                download_mc_btn = gr.DownloadButton("📥 Download Monte Carlo Iteration Dataset (CSV)", size="sm")
            
            gr.Markdown("*Edit the `Start_Date` (YYYY-MM-DD) or `Rig_Assigned` directly in this table, then click **Apply Manual Edits**. The Gantt and Map will instantly redraw.*")
            
            table_output = gr.Dataframe(label="Editable Sequence Table", interactive=True)
            apply_manual_btn = gr.Button("🔄 Apply Manual Edits to Charts", variant="secondary")

    # --- UI LOGIC CALLBACKS ---
    def process_well_file(file, df_curr):
        if file is None: return df_curr, gr.update(), gr.update()
        try:
            file_path = file if isinstance(file, str) else file.name
            if file_path.endswith('.csv'): new_df = pd.read_csv(file_path)
            else: new_df = pd.read_excel(file_path)
            merged = pd.concat([df_curr, new_df], ignore_index=True).drop_duplicates(subset=["Well_ID"])
            ids = merged["Well_ID"].tolist()
            gr.Info(f"✅ Loaded {len(new_df)} wells successfully!")
            return merged, gr.update(choices=ids), gr.update(choices=ids)
        except Exception as e:
            gr.Warning(f"⚠️ Error reading well file: {str(e)}")
            return df_curr, gr.update(), gr.update()

    def process_rig_file(file, fleet_curr):
        if file is None: return fleet_curr, gr.update()
        try:
            file_path = file if isinstance(file, str) else file.name
            if file_path.endswith('.csv'): new_df = pd.read_csv(file_path)
            else: new_df = pd.read_excel(file_path)
            
            updated_fleet = fleet_curr.copy()
            default_colors = ["#1f77b4", "#ff7f0e", "#d62728", "#2ca02c", "#9467bd", "#e377c2"]
            
            for idx, row in new_df.iterrows():
                name = str(row.get("Rig_Name", "")).strip()
                if not name: continue
                r_type = str(row.get("Type", "Light Unit"))
                cats_raw = str(row.get("Supported_Job_Categories", ""))
                cats = parse_categories(cats_raw, ALL_KNOWN_CATEGORIES)
                mob_days = int(row.get("Mob_Demob_Days", 2))
                cost = float(row.get("Daily_Cost_kUSD", 12))
                contract = str(row.get("Contract_End_Date", (BASE_DATE + timedelta(days=365)).strftime("%Y-%m-%d")))
                color = default_colors[len(updated_fleet) % len(default_colors)]
                
                updated_fleet[name] = {
                    "Type": r_type,
                    "Supported_Job_Categories": cats,
                    "Mob_Demob_Days": mob_days,
                    "Daily_Cost_kUSD": cost,
                    "Contract_End_Date": contract,
                    "Color": color
                }
            gr.Info(f"✅ Loaded {len(new_df)} rigs successfully!")
            return updated_fleet, gr.update(value=f"Rig-{len(updated_fleet)+1} (New Unit)")
        except Exception as e:
            gr.Warning(f"⚠️ Error reading rig file: {str(e)}")
            return fleet_curr, gr.update()

    def generate_well_template_csv():
        template_df = pd.DataFrame([{
            "Well_ID": "WELL-X",
            "Job Category": "Slickline Services",
            "Duration_Days": 10,
            "BOPD": 500,
            "Lat": 4.15,
            "Lon": 112.25
        }])
        path = "well_template.csv"
        template_df.to_csv(path, index=False)
        return path

    def generate_rig_template_csv():
        template_df = pd.DataFrame([{
            "Rig_Name": "Rig-1 (Light Unit)",
            "Type": "Light Unit",
            "Supported_Job_Categories": "Downhole Logging, Survey & Test, Wellhead Maintenance, Slickline Services",
            "Mob_Demob_Days": 2,
            "Daily_Cost_kUSD": 12,
            "Contract_End_Date": (BASE_DATE + timedelta(days=365)).strftime("%Y-%m-%d")
        }])
        path = "rig_template.csv"
        template_df.to_csv(path, index=False)
        return path

    def inject_well(df, w_id, j_cat, dur, gain, lat, lon):
        new_row = {
            "Well_ID": w_id, "Job Category": j_cat, 
            "Duration_Days": dur, "BOPD": gain, 
            "Lat": lat, "Lon": lon
        }
        updated = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        ids = updated["Well_ID"].tolist()
        gr.Info(f"✅ Injected well {w_id} successfully!")
        return updated, gr.update(choices=ids), gr.update(choices=ids)

    def inject_rig(fleet, name, r_type, cats, mob_days, cost, contract):
        default_colors = ["#1f77b4", "#ff7f0e", "#d62728", "#2ca02c", "#9467bd", "#e377c2"]
        color = default_colors[len(fleet) % len(default_colors)]
        updated = fleet.copy()
        updated[name] = {
            "Type": r_type, 
            "Supported_Job_Categories": cats, 
            "Mob_Demob_Days": mob_days,
            "Daily_Cost_kUSD": cost, 
            "Color": color,
            "Contract_End_Date": contract
        }
        gr.Info(f"✅ Added {name} to the fleet successfully!")
        return updated, gr.update(value=f"Rig-{len(updated)+1} (New Unit)")

    def manual_override(edited_df, raw_candidates, fleet_specs):
        if edited_df is None or edited_df.empty:
            return "No active schedule.", go.Figure(), go.Figure(), go.Figure(), pd.DataFrame(), pd.DataFrame()
        if edited_df["Rig_Assigned"].isna().any():
            return "⚠️ Every manually edited row needs a rig assignment.", go.Figure(), go.Figure(), go.Figure(), pd.DataFrame(), pd.DataFrame()

        unknown_rigs = set(edited_df["Rig_Assigned"].dropna()) - set(fleet_specs)
        if unknown_rigs:
            names = ", ".join(sorted(map(str, unknown_rigs)))
            return f"⚠️ Unknown rig assignment: {names}.", go.Figure(), go.Figure(), go.Figure(), pd.DataFrame(), pd.DataFrame()

        try:
            merged = edited_df.merge(raw_candidates[["Well_ID", "Lat", "Lon"]], on="Well_ID", how="left")
            merged['Start_Date'] = pd.to_datetime(merged['Start_Date'], errors="raise")
            merged['End_Date'] = pd.to_datetime(merged['End_Date'], errors="raise")
        except (KeyError, TypeError, ValueError) as exc:
            return f"⚠️ Manual edits could not be applied: {exc}", go.Figure(), go.Figure(), go.Figure(), pd.DataFrame(), pd.DataFrame()
        if merged[["Start_Date", "End_Date", "Lat", "Lon"]].isna().any().any():
            return "⚠️ Manual edits need valid dates and Well_ID values from the candidate dataset.", go.Figure(), go.Figure(), go.Figure(), pd.DataFrame(), pd.DataFrame()
        
        resolved_rows = []
        
        for rig in merged['Rig_Assigned'].unique():
            rig_df = merged[merged['Rig_Assigned'] == rig].sort_values(by="Start_Date").copy()
            mob_demob = _as_nonnegative_int(fleet_specs[rig]["Mob_Demob_Days"], f"{rig} Mob/Demob Days")
            daily_cost_kusd = fleet_specs[rig]["Daily_Cost_kUSD"]
            
            prev_end = None
            prev_lat = None
            prev_lon = None
            
            for idx, row in rig_df.iterrows():
                total_duration = max((row["End_Date"] - row["Start_Date"]).days, 1)
                work_duration = max(total_duration - mob_demob, 1)
                row["Duration_Days"] = work_duration
                row["Mob_Demob_Days"] = mob_demob
                
                if prev_end is None:
                    start_date = row["Start_Date"]
                else:
                    dist_km = haversine(prev_lat, prev_lon, row["Lat"], row["Lon"])
                    transit_days = int(np.ceil(dist_km / 50.0)) + 1
                    earliest_start = prev_end + timedelta(days=transit_days)
                    start_date = earliest_start
                
                end_date = start_date + timedelta(days=work_duration + mob_demob)
                
                row["Start_Date"] = start_date
                row["End_Date"] = end_date
                cost_kusd = _well_cost_units(work_duration, mob_demob, daily_cost_kusd) / COST_UNIT_SCALE
                row["Cost_kUSD"] = cost_kusd
                row["BOPD_per_kUSD"] = round(float(row["BOPD"]) / cost_kusd, 4) if cost_kusd > 0 else 0.0
                row["Contract_End_Date"] = fleet_specs[rig].get("Contract_End_Date", "N/A")
                
                prev_end = end_date
                prev_lat = row["Lat"]
                prev_lon = row["Lon"]
                resolved_rows.append(row)
                
        resolved_df = pd.DataFrame(resolved_rows)
        resolved_df["Start_Date"] = resolved_df["Start_Date"].dt.strftime("%Y-%m-%d")
        resolved_df["End_Date"] = resolved_df["End_Date"].dt.strftime("%Y-%m-%d")
        
        return render_visuals(resolved_df, raw_candidates, fleet_specs)

    def sync_locks_from_drops(dropped_list, candidate_df):
        all_wells = candidate_df["Well_ID"].tolist()
        dropped = dropped_list or []
        available_locks = [w for w in all_wells if w not in dropped]
        return gr.update(choices=available_locks)

    def sync_drops_from_locks(locked_list, candidate_df):
        all_wells = candidate_df["Well_ID"].tolist()
        locked = locked_list or []
        available_drops = [w for w in all_wells if w not in locked]
        return gr.update(choices=available_drops)

    demo.load(lambda df: (gr.update(choices=df["Well_ID"].tolist()), gr.update(choices=df["Well_ID"].tolist())), inputs=candidate_state, outputs=[dropped_wells, locked_wells])
    
    dropped_wells.change(fn=sync_locks_from_drops, inputs=[dropped_wells, candidate_state], outputs=[locked_wells])
    locked_wells.change(fn=sync_drops_from_locks, inputs=[locked_wells, candidate_state], outputs=[dropped_wells])

    file_upload_wells.change(fn=process_well_file, inputs=[file_upload_wells, candidate_state], outputs=[candidate_state, dropped_wells, locked_wells])
    file_upload_rigs.change(fn=process_rig_file, inputs=[file_upload_rigs, fleet_state], outputs=[fleet_state, new_rig_name])
    
    download_well_template_btn.click(fn=generate_well_template_csv, outputs=[download_well_template_btn])
    download_rig_template_btn.click(fn=generate_rig_template_csv, outputs=[download_rig_template_btn])
    
    add_well_btn.click(fn=inject_well, inputs=[candidate_state, new_id, new_cat, new_dur, new_gain, new_lat, new_lon], outputs=[candidate_state, dropped_wells, locked_wells])
    add_rig_btn.click(fn=inject_rig, inputs=[fleet_state, new_rig_name, new_rig_type, new_rig_cats, new_rig_mob, new_rig_cost, new_rig_contract], outputs=[fleet_state, new_rig_name])
    
    run_btn.click(
        fn=lambda: False, outputs=[is_approved_state]
    ).then(
        fn=run_optimization_wrapper,
        inputs=[candidate_state, fleet_state, dropped_wells, locked_wells, table_output, is_approved_state],
        outputs=[run_btn, confirm_group, confirm_msg, kpi_output, gantt_output, map_output, mc_output, mc_data_state, table_output]
    )

    approve_btn.click(
        fn=lambda: True, outputs=[is_approved_state]
    ).then(
        fn=run_optimization_wrapper,
        inputs=[candidate_state, fleet_state, dropped_wells, locked_wells, table_output, is_approved_state],
        outputs=[run_btn, confirm_group, confirm_msg, kpi_output, gantt_output, map_output, mc_output, mc_data_state, table_output]
    )

    deny_btn.click(
        fn=lambda: (gr.update(visible=True), gr.update(visible=False), gr.update(value=[])),
        outputs=[run_btn, confirm_group, dropped_wells]
    )
    
    apply_manual_btn.click(
        fn=manual_override, 
        inputs=[table_output, candidate_state, fleet_state], 
        outputs=[kpi_output, gantt_output, map_output, mc_output, mc_data_state, table_output]
    )

    download_mc_btn.click(
        fn=export_mc_csv,
        inputs=[mc_data_state],
        outputs=[download_mc_btn]
    )

demo.launch(inline=True, share=False)

/Users/omarhambaran/softwareProducts/bayu-well-intervention-scheduling/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
